# 端到端 Profiling

当前 `torchtitan-npu` 可直接对比三条 Wordle 路线：

| 配置 | Packing | Attention |
|---|---|---|
| `sft_qwen3_1_7b_wordle` | non-greedy | causal SDPA |
| `sft_qwen3_1_7b_wordle_block_causal_sdpa` | greedy | block-causal SDPA |
| `sft_qwen3_1_7b_wordle_tnd` | greedy | NPU Varlen Attention |

三次运行都使用 2 卡、`seq_len=4096`、`global_batch_size=4`，并采集 rank 0 的 Step 5。


In [ ]:
import os
from pathlib import Path

original_dir = Path.cwd()
configured_root = os.environ.get('TORCHTITAN_ROOT')
candidates = [Path(configured_root)] if configured_root else []
for parent in (original_dir, *original_dir.parents):
    candidates.extend((parent / 'torchtitan-npu', parent.parent / 'torchtitan-npu'))
torchtitan_root = next((path.resolve() for path in candidates if (path / 'scripts/run_train.sh').is_file()), None)
if torchtitan_root is None:
    raise RuntimeError('未找到 torchtitan-npu；请设置 TORCHTITAN_ROOT。')
os.chdir(torchtitan_root)
if cann_env := os.environ.get('CANN_ENV_SCRIPT'):
    os.environ['BASH_ENV'] = cann_env
print('torchtitan root:', torchtitan_root)

## Baseline：causal SDPA


In [ ]:
%%bash
set -euo pipefail
# 清理上次运行的 baseline profiling
rm -rf outputs/checkpoints/baseline_s4096 outputs/profile_traces/baseline_s4096
NGPU=2 \
DATASET_PATH=./assets/data/wordle \
MODULE=torchtitan_npu.models.qwen3 \
CONFIG=sft_qwen3_1_7b_wordle \
bash scripts/run_train.sh \
  --training.steps 10 \
  --training.global-batch-size 4 \
  --checkpoint.enable \
  --checkpoint.load-only \
  --checkpoint.folder checkpoints/baseline_s4096 \
  --training.seq_len 4096 \
  --profiling.enable-profiling \
  --profiling.profile-ranks 0 \
  --profiling.profile-step-start 5 \
  --profiling.profile-step-end 6 \
  --profiling.profile-with-memory \
  --profiling.save-traces-folder profile_traces/baseline_s4096 \
dataloader:chat_data_loader_config \
  --dataloader.dataset_path "assets/data/wordle"


## Block-causal SDPA


In [ ]:
%%bash
set -euo pipefail
# 清理上次运行的 block-causal profiling
rm -rf outputs/checkpoints/block_causal_sdpa_s4096 outputs/profile_traces/block_causal_sdpa_s4096
NGPU=2 \
DATASET_PATH=./assets/data/wordle \
MODULE=torchtitan_npu.models.qwen3 \
CONFIG=sft_qwen3_1_7b_wordle_block_causal_sdpa \
bash scripts/run_train.sh \
  --training.steps 10 \
  --training.global-batch-size 4 \
  --checkpoint.enable \
  --checkpoint.load-only \
  --checkpoint.folder checkpoints/block_causal_sdpa_s4096 \
  --training.seq_len 4096 \
  --profiling.enable-profiling \
  --profiling.profile-ranks 0 \
  --profiling.profile-step-start 5 \
  --profiling.profile-step-end 6 \
  --profiling.profile-with-memory \
  --profiling.save-traces-folder profile_traces/block_causal_sdpa_s4096 \
dataloader:chat_data_loader_config \
  --dataloader.dataset_path "assets/data/wordle"


## NPU Varlen Attention


In [ ]:
%%bash
set -euo pipefail
# 清理上次运行的 TND profiling
rm -rf outputs/checkpoints/varlen_tnd_s4096 outputs/profile_traces/varlen_tnd_s4096
NGPU=2 \
DATASET_PATH=./assets/data/wordle \
MODULE=torchtitan_npu.models.qwen3 \
CONFIG=sft_qwen3_1_7b_wordle_tnd \
bash scripts/run_train.sh \
  --training.steps 10 \
  --training.global-batch-size 4 \
  --checkpoint.enable \
  --checkpoint.load-only \
  --checkpoint.folder checkpoints/varlen_tnd_s4096 \
  --training.seq_len 4096 \
  --profiling.enable-profiling \
  --profiling.profile-ranks 0 \
  --profiling.profile-step-start 5 \
  --profiling.profile-step-end 6 \
  --profiling.profile-with-memory \
  --profiling.save-traces-folder profile_traces/varlen_tnd_s4096 \
dataloader:chat_data_loader_config \
  --dataloader.dataset_path "assets/data/wordle"


## 输出

Trace 保存到 `outputs/profile_traces/{baseline_s4096,block_causal_sdpa_s4096,varlen_tnd_s4096}`。先确认本次运行使用的是“每条样本的位置编号从 0 重新开始”的边界实现，再比较 Step 5 的 Attention 算子、device time 和 peak memory。旧实现按 EOS 切分样本，它生成的 trace 不能与新实现混用。


## 配置方式

| 路线 | Model spec | Packing | Mask / metadata | Trace 中的 Attention API |
|---|---|---|---|---|
| SDPA + `is_causal` | 原 `ScaledDotProductAttention`，`mask_type="causal"` | non-greedy，每条 sample pad 到 4096 | `is_causal=True` | `aclnnFlashAttentionScore` |
| SDPA + block-causal mask | inner attention 不变，只把每层 `mask_type` 设为 `block_causal` | greedy | bool mask `[2,1,4096,4096]`，`attn_mask=mask`、`is_causal=False` | `aclnnFlashAttentionScore` |
| TND Varlen + block-causal | `_enable_npu_varlen_attention()` 把每层 inner attention 换成 `NPUVarlenAttention.Config()` | greedy | 从样本起点生成 `cu_seqlens`，BSND→TND，`sparse_mode=7` | `aclnnFlashAttentionVarLenScore` / `aclnnFlashAttentionUnpaddingScoreGrad` |

两条 greedy 路线使用同一批样本起点：SDPA 把它们展开成 `[B,1,S,S]` 掩码，Varlen 把它们转换成累计长度。本轮 trace 中，causal/block 的 Q shape 为 `[2,16,4096,128]`，block 额外传入 `[2,1,4096,4096]` bool mask；TND 的 Q shape 为 `[8192,16,128]`。


## Step 5 算子结果

2026-07-29 本轮结果来自同一 checkpoint、seed、数据顺序、两卡硬件和 Step 5。旧 trace 把多轮对话按 EOS 拆段，不能与本轮 position-boundary trace 混用。selective activation checkpointing 使被采样 step 包含 56 次 Attention forward（含重计算）和 28 次 backward。

| 指标 | SDPA causal | SDPA block mask | TND Varlen |
|---|---:|---:|---:|
| Forward 平均时间 / call | 1.397 ms | 2.578 ms | 0.625 ms |
| Backward 平均时间 / call | 3.093 ms | 4.861 ms | 1.575 ms |
| Attention device total / step | 164.815 ms | 280.452 ms | 79.125 ms |
| Stage time | 1.683 s | 1.803 s | 1.638 s |
| 训练日志 max reserved | 28.25 GiB | 28.48 GiB | 26.74 GiB |
| Trace-window max reserved | 32.01 GiB | 32.22 GiB | 30.48 GiB |
| Step 5 原始样本数（全局） | 4 | 12 | 12 |

Block mask 修复了 greedy packing 的语义，但仍走 dense SDPA：相对 causal，Attention device total 增加 70.2%，Stage time 增加 7.1%。TND 相对 block mask 把 Attention device total 降低 71.8%、Stage time 降低 9.1%，训练日志 max reserved 少 1.74 GiB（6.1%）。

排除首次编译和 profiler 解析开销后，steps 7–10 的均值如下：

| 指标 | SDPA causal | SDPA block mask | TND Varlen |
|---|---:|---:|---:|
| Step time | 1.679 s | 1.813 s | 1.637 s |
| slot TPS/device | 4,880 | 4,519 | 5,005 |

因此 TND 相对语义等价的 block-SDPA，稳态 step time 降低 9.7%，slot TPS/device 提高 10.7%。两条 greedy 路线 10 步的 loss/grad norm 保持接近；最大 loss 差为 0.00488（首次编译 step），steps 2–10 不超过 0.00107。该证据支持 backend 数值一致性和当前 workload 的执行收益，不证明长期收敛速度。


## TPS 口径

当前 TorchTitan 在 `Trainer.batch_generator()` 中累加 `labels.numel()`，再用它计算日志里的 `tps`。本节每卡的分子固定为 `local_batch_size × seq_len = 2 × 4096 = 8192`：padding、prompt 和真实训练 token 都被等价计数。因此日志 `tps` 只是 **slot TPS / device**，greedy packing 不会提高其分子。

不应直接替换该指标，因为 TFlops/MFU 仍需要物理 slot 吞吐。建议重命名并同时报告：

| 指标 | 分子 | 用途 |
|---|---|---|
| `slot_tps/device` | `labels.numel()` | 计算吞吐、TFlops/MFU |
| `effective_input_tps` | padding 前的 input token 数 | 衡量 packing 后的有效数据吞吐 |
| `supervised_tps` | `labels != -100` 的数量 | 衡量实际贡献 loss 的 token 吞吐 |
| `raw_sample/s` | 每批原始 sample 数 | 直接表达 non-greedy→greedy 收益 |

`supervised_tps` 可以直接从 labels 统计。位置编号能够恢复训练区间，却不能可靠地区分真实样本和最后补齐容器的 padding，也不能给出 padding 前的业务 token 数；因此 `effective_input_tps` 和 `raw_sample/s` 最好由 DataLoader 直接计数。不能用 EOS 或 `labels != -100` 反推：真实消息也有 EOS，prompt label 也可能是 `-100`。在当前两卡纯 FSDP 配置下，全局业务吞吐应先对 DP ranks 求和分子，再除以 wall time。


In [ ]:
%cd $original_dir

## 练习

1. （单选题）06.06 的三条受控路线是什么？
    A. causal SDPA、dense block-causal SDPA、NPU TND VarLen
    B. DDP、TP、PP
    C. eager、量化、推理
    D. FSDP、CP、EP

2. （判断题）Profiler trace 负责证明实际算子路径；关闭 profiler 的重复 wall-time 更适合回答稳定的端到端速度。

3. （判断题）VarLen 跳过跨区间 Attention pair 后，padding 位置的 embedding、MLP 和 Norm 也会自动全部跳过。

4. （单选题）解释 packing 的端到端收益时，为什么不能只报告原生 TPS？
    A. 原生 TPS 只统计容器位置，可能掩盖 padding 与 supervised token 比例
    B. 原生 TPS 不包含任何 token
    C. 原生 TPS 只能在 CPU 上计算
    D. 原生 TPS 与 step time 无关

In [ ]:
!cat ./answer/06.06_answer.txt
